In [1]:
import shap
# pip install xgboost, catboost, lightgbm
import xgboost
import lightgbm

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score, precision_score
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV
from sklearn.impute import SimpleImputer, KNNImputer
from imblearn.pipeline import Pipeline
#from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.datasets import make_classification
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [2]:
df = pd.read_csv('student_dropout_dataset_v3.csv')
df.head(5)

,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,4,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


In [3]:
df.describe()

,Student_ID,Age,Family_Income,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Stress_Index,GPA,Semester_GPA,CGPA,Dropout
count,10000.00000,10000.00000,9500.000000,9500.000000,10000.00000,10000.000000,10000.00000,9500.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,21.02606,38377.247474,4.014592,81.73683,1.799700,30.17926,5.507147,2.308440,2.300057,2.298761,0.23540
std,2886.89568,2.13981,20496.232179,1.295450,8.22093,1.344307,11.91887,1.765951,1.061717,1.074407,1.072555,0.42427
min,1.00000,17.00000,25000.000000,0.500000,38.20000,0.000000,5.00000,1.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,19.50000,25000.000000,3.160000,76.40000,1.000000,21.90000,4.300000,1.550000,1.520000,1.520000,0.00000
50%,5000.50000,21.00000,29740.500000,4.000000,81.80000,2.000000,30.20000,5.500000,2.350000,2.350000,2.350000,0.00000
75%,7500.25000,22.50000,44520.000000,4.870000,87.30000,3.000000,38.40000,6.700000,3.120000,3.150000,3.150000,0.00000
max,10000.00000,29.60000,316601.000000,8.980000,100.00000,8.000000,74.90000,10.000000,4.000000,4.000000,4.000000,1.00000


In [4]:
df.shape

(10000, 19)

In [5]:
df.dropna(inplace=True)

In [6]:
df.shape

(9020, 19)

In [7]:
df = df.drop(columns=['Student_ID'])


In [8]:
df.shape

(9020, 18)

In [10]:
df.head(5)

,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
4,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0
6,24.5,Male,25000.0,Yes,3.00,78.2,1,37.4,Yes,Yes,7.3,0.64,0.33,0.44,Year 4,CS,Bachelor,0


In [9]:
df["Internet_Access"] = df["Internet_Access"].replace({"Yes": 1, "No": 0})
df["Part_Time_Job"]   = df["Part_Time_Job"].replace({"Yes": 1, "No": 0})
df["Scholarship"]     = df["Scholarship"].replace({"Yes": 1, "No": 0})
df["Gender"]          = df["Gender"].replace({"Male": 1, "Female": 0})

C:\Users\aikat\AppData\Local\Temp\ipykernel_14764\2599807966.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Internet_Access"] = df["Internet_Access"].replace({"Yes": 1, "No": 0})
C:\Users\aikat\AppData\Local\Temp\ipykernel_14764\2599807966.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Part_Time_Job"]   = df["Part_Time_Job"].replace({"Yes": 1, "No": 0})
C:\Users\aikat\AppData\Local\Temp\ipykernel_14764\2599807966.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future 

In [10]:
df

,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,22.1,1,25000.0,1,3.36,86.1,2,20.4,1,0,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,20.7,1,25000.0,1,4.30,68.0,2,44.0,0,0,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,22.4,1,40183.0,1,4.40,70.9,0,48.9,1,0,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
4,20.5,0,25319.0,1,4.19,75.7,1,23.0,0,0,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0
6,24.5,1,25000.0,1,3.00,78.2,1,37.4,1,1,7.3,0.64,0.33,0.44,Year 4,CS,Bachelor,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,23.9,0,42286.0,0,4.62,92.0,0,10.0,1,1,5.5,1.60,0.99,0.97,Year 2,Arts,Bachelor,0
9996,17.0,0,61103.0,1,2.87,75.2,3,32.4,0,1,6.7,3.09,3.09,3.09,Year 1,Business,Master,1
9997,19.4,1,25000.0,1,4.73,74.9,4,25.4,0,0,3.5,3.45,3.37,3.43,Year 4,Business,Bachelor,0
9998,22.1,0,40302.0,1,5.85,74.2,1,5.0,0,1,6.2,3.35,3.34,3.34,Year 1,CS,High School,0


In [14]:
ohe_cols = ['Semester', 'Department', 'Parental_Education']
df_encoded = pd.get_dummies(df, columns=ohe_cols, drop_first=True).astype(int)

In [17]:
df_encoded.shape

(9020, 25)

In [18]:
df_encoded

,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,...,Semester_Year 2,Semester_Year 3,Semester_Year 4,Department_Business,Department_CS,Department_Engineering,Department_Science,Parental_Education_High School,Parental_Education_Master,Parental_Education_PhD
0,22,1,25000,1,3,86,2,20,1,0,...,0,0,0,0,0,0,0,1,0,0
1,20,1,25000,1,4,68,2,44,0,0,...,0,1,0,0,0,1,0,0,0,0
2,22,1,40183,1,4,70,0,48,1,0,...,0,0,0,0,0,0,0,0,1,0
4,20,0,25319,1,4,75,1,23,0,0,...,0,0,1,1,0,0,0,0,0,0
6,24,1,25000,1,3,78,1,37,1,1,...,0,0,1,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,23,0,42286,0,4,92,0,10,1,1,...,1,0,0,0,0,0,0,0,0,0
9996,17,0,61103,1,2,75,3,32,0,1,...,0,0,0,1,0,0,0,0,1,0
9997,19,1,25000,1,4,74,4,25,0,0,...,0,0,1,1,0,0,0,0,0,0
9998,22,0,40302,1,5,74,1,5,0,1,...,0,0,0,0,1,0,0,1,0,0


In [19]:
df_encoded.to_csv("df_cleaned.csv", index=False)